# EMA Touch-and-Rejection

EMA touch-and-rejection — a pullback strategy ported from the standalone `ema` project. It waits for price to *pull back into* an EMA and *reject* it: a bar whose wick tags the EMA (within a `delta` tolerance) but which closes back on the trend side. Trades both directions; long is tried first on an ambiguous bar and the run's direction gate decides which sides are allowed.

__How the EMA Touch Algorithm Determines Entry/Exit:__
- One entry EMA (`ema_touch_period`, default 50); an optional slower regime EMA can gate each side.
- Long Entry: the bar's **low** comes within `delta` of the EMA AND the bar **closes >= EMA** (rejection up). With a regime filter: only if close > the regime EMA.
- Short Entry: the bar's **high** comes within `delta` of the EMA AND the bar **closes <= EMA** (rejection down).
- `delta` units follow `ema_touch_delta_mode`: `absolute` (quote points), `percent` (% of the EMA), or `atr` (× ATR, cross-symbol).
- Exit: the injected exit policy — default a 1% fixed stop + a 3R take-profit, stop-first on an ambiguous bar (both checked intrabar).
- An entry whose stop would land on the wrong side of the fill is skipped.

## Configuration

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## EMA Touch-and-Rejection

In [ ]:
# Import EMA touch-and-rejection strategy
from engine.strategies import EmaTouchStrategy

In [ ]:
# Backtest EMA touch-and-rejection strategy.
# Key signal knobs (StrategyConfig): ema_touch_period (EMA span), ema_touch_delta
# + ema_touch_delta_mode ('absolute' quote-points | 'percent' | 'atr'), and the
# optional ema_touch_regime_filter. The 40-point absolute default suits BTCUSDT;
# switch to delta_mode='atr' for cross-symbol use. Stop/TP come from the exit
# policy (default 1% stop + 3R target).
config = StrategyConfig()
strategy = EmaTouchStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# EMA touch-and-rejection strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()